# Consolidação dos dados

In [ ]:
# pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Imports

In [2]:
import re
from pathlib import Path
import pandas as pd

### Função de consolidação

In [4]:
# Defina o caminho das pastas aqui
PASTA_ORIGEM = Path("./raw")  # Onde estão os CSVs brutos
PASTA_DESTINO = Path(
    "./dados_consolidados"
)  # Onde serão salvos os arquivos finais
PASTA_DESTINO.mkdir(parents=True, exist_ok=True)

In [5]:
def consolidar_categoria(nome_categoria, termos_busca):
  """Filtra os CSVs correspondentes à categoria e anos (2021-2025), junta os dados

  e salva um arquivo CSV consolidado na pasta de destino.
  """
  anos_alvo = set(range(2021, 2026))  # 2021 a 2025
  arquivos_encontrados = []

  # Procura os arquivos na pasta
  for arquivo in PASTA_ORIGEM.glob("*.csv"):
    nome_lc = arquivo.name.lower()

    # Checa se possui algum termo da categoria
    tem_categoria = any(termo in nome_lc for termo in termos_busca)

    # Extrai o ano do nome do arquivo
    match_ano = re.search(r"\b(2021|2022|2023|2024|2025)\b", nome_lc)
    ano = int(match_ano.group(1)) if match_ano else None

    if tem_categoria and ano in anos_alvo:
      arquivos_encontrados.append((arquivo, ano))

  if not arquivos_encontrados:
    print(f"⚠️  Nenhum arquivo encontrado para: {nome_categoria}")
    return None

  print(
    f"🔄 Consolidando '{nome_categoria.upper()}' ({len(arquivos_encontrados)} arquivos encontrados)..."
  )
  dfs = []

  # Ordena por ano
  for caminho, ano in sorted(arquivos_encontrados, key=lambda x: x[1]):
    try:
      # sep=None detecta automaticamente vírgula (,) ou ponto e vírgula (;)
      df = pd.read_csv(caminho, sep=None, engine="python")

      # Adiciona metadados úteis
      df["ano_referencia"] = ano
      df["arquivo_origem"] = caminho.name

      dfs.append(df)
      print(f"  ├─ Carregado: {caminho.name} ({len(df):,} linhas)")
    except Exception as e:
      print(f"  ❌ Erro ao ler {caminho.name}: {e}")

  if dfs:
    df_final = pd.concat(dfs, ignore_index=True)
    caminho_saida = PASTA_DESTINO / f"{nome_categoria}_2021_2025.csv"

    # Salva o arquivo final
    df_final.to_csv(caminho_saida, index=False, encoding="utf-8-sig")
    print(
      f"✅ CONSOLIDAÇÃO CONCLUÍDA: {caminho_saida} | Total: {len(df_final):,} linhas\n"
    )
    return df_final

### Licitações

In [5]:
# Bloco para Licitações (inclui variações com e sem acento)
df_licitacoes = consolidar_categoria(
    "licitacoes", ["licitaca", "licitaco", "licita"]
)

# Exibe as primeiras linhas no Jupyter
if df_licitacoes is not None:
  display(df_licitacoes.head())

🔄 Consolidando 'LICITACOES' (5 arquivos encontrados)...
  ├─ Carregado: licitacoes-2021.csv (38,603 linhas)
  ├─ Carregado: licitacoes-2022.csv (42,590 linhas)
  ├─ Carregado: licitacoes-2023.csv (45,454 linhas)
  ├─ Carregado: licitacoes-2024.csv (43,564 linhas)
  ├─ Carregado: licitacoes-2025.csv (49,909 linhas)
✅ CONSOLIDAÇÃO CONCLUÍDA: dados_consolidados\licitacoes_2021_2025.csv | Total: 220,120 linhas



,﻿nome_municipio,codigo_unidade_gestora,descricao_unidade_gestora,numero_licitacao,numero_protocolo_tce,ano_licitacao,modalidade,objeto_licitacao,data_homologacao,nome_proponente,cpf_cnpj_proponente,valor_ofertado,situacao_proposta,ano_referencia,arquivo_origem
0,Campina Grande,603050,Fundo Municipal de Assistencia Social de Campi...,25010/2021,Doc. 52818/21,2021,Pregão Eletrônico (Lei Nº 10.520/2002),"AQUISIÇÃO DE MATERIAL DE LIMPEZA, DESCARTÁVEIS...",20/08/2021,Oliveira & Eulálio Produtos de Limpeza Ltda - ME,7324070000144,704.376,Vencedora,2021,licitacoes-2021.csv
1,Campina Grande,603050,Fundo Municipal de Assistencia Social de Campi...,25010/2021,Doc. 52818/21,2021,Pregão Eletrônico (Lei Nº 10.520/2002),"AQUISIÇÃO DE MATERIAL DE LIMPEZA, DESCARTÁVEIS...",20/08/2021,Dental Higix Produtos Odontologicos Medicos Ho...,26240632000116,194.560,Vencedora,2021,licitacoes-2021.csv
2,Campina Grande,603050,Fundo Municipal de Assistencia Social de Campi...,25010/2021,Doc. 52818/21,2021,Pregão Eletrônico (Lei Nº 10.520/2002),"AQUISIÇÃO DE MATERIAL DE LIMPEZA, DESCARTÁVEIS...",20/08/2021,SUPRIMAIS COMERCIO E SERVICOS DE INFORMATICA LTDA,9004901000126,"538.034,2",Vencedora,2021,licitacoes-2021.csv
3,Campina Grande,603050,Fundo Municipal de Assistencia Social de Campi...,25010/2021,Doc. 52818/21,2021,Pregão Eletrônico (Lei Nº 10.520/2002),"AQUISIÇÃO DE MATERIAL DE LIMPEZA, DESCARTÁVEIS...",20/08/2021,Machado Armarinhos Ltda,24174062000188,400.500,Vencedora,2021,licitacoes-2021.csv
4,Campina Grande,603050,Fundo Municipal de Assistencia Social de Campi...,25010/2021,Doc. 52818/21,2021,Pregão Eletrônico (Lei Nº 10.520/2002),"AQUISIÇÃO DE MATERIAL DE LIMPEZA, DESCARTÁVEIS...",20/08/2021,BIDDEN COMERCIAL LTDA,36181473000180,122.000,Vencedora,2021,licitacoes-2021.csv


### Servidores

In [ ]:
# Bloco para Servidores (inclui folha de pagamento)
df_servidores = consolidar_categoria(
    "servidores", ["servidor", "servidores", "folha", "pessoal"]
)
# Exibe as primeiras linhas no Jupyter
if df_servidores is not None:
  display(df_servidores.head())

### Receitas

In [6]:
# Bloco para Receitas
df_receitas = consolidar_categoria("receitas", ["receita", "receitas"])

# Exibe as primeiras linhas no Jupyter
if df_receitas is not None:
  display(df_receitas.head())

🔄 Consolidando 'RECEITAS' (5 arquivos encontrados)...
  ├─ Carregado: receitas-2021.csv (140,330 linhas)
  ├─ Carregado: receitas-2022.csv (143,317 linhas)
  ├─ Carregado: receitas-2023.csv (156,440 linhas)
  ├─ Carregado: receitas-2024.csv (160,527 linhas)
  ├─ Carregado: receitas-2025.csv (163,252 linhas)
✅ CONSOLIDAÇÃO CONCLUÍDA: dados_consolidados\receitas_2021_2025.csv | Total: 763,866 linhas



,﻿municipio,codigo_unidade_gestora,descricao_unidade_gestora,mes_ano,ano,codigo_receita,descricao_receita,tipo_atualizacao_receita,valor,codigo_fonte_recurso,descricao_fonte_recurso,co,descricao_co,ano_referencia,arquivo_origem
0,Água Branca,201001,Prefeitura Municipal de Água Branca,62021,2021,24181091,Outras Transferências de Convênios da União - ...,Lançamento de Receita,"135.238,09",1510,Outras Transferências de Convênios ou Contrato...,Não,Não Aplicável,2021,receitas-2021.csv
1,Água Branca,201001,Prefeitura Municipal de Água Branca,62021,2021,24181051,Transferências de Convênios da União destinada...,Lançamento de Receita,96.920,1510,Outras Transferências de Convênios ou Contrato...,Não,Não Aplicável,2021,receitas-2021.csv
2,Água Branca,201001,Prefeitura Municipal de Água Branca,32021,2021,24181051,Transferências de Convênios da União destinada...,Lançamento de Receita,"206.724,95",1510,Outras Transferências de Convênios ou Contrato...,Não,Não Aplicável,2021,receitas-2021.csv
3,Água Branca,201001,Prefeitura Municipal de Água Branca,72021,2021,24180511,Prog. de Apoio ao Transp. Escolar para Educaçã...,Lançamento de Receita,189.900,1125,Transferências de Convênios - Educação - Recur...,Não,Não Aplicável,2021,receitas-2021.csv
4,Água Branca,201001,Prefeitura Municipal de Água Branca,42021,2021,24180511,Prog. de Apoio ao Transp. Escolar para Educaçã...,Lançamento de Receita,193.632,1125,Transferências de Convênios - Educação - Recur...,Não,Não Aplicável,2021,receitas-2021.csv


### Despesas

In [6]:
# Bloco para Despesas
df_despesas = consolidar_categoria("despesas", ["despesa", "despesas"])

# Exibe as primeiras linhas no Jupyter
if df_despesas is not None:
  display(df_despesas.head())

🔄 Consolidando 'DESPESAS' (5 arquivos encontrados)...
  ├─ Carregado: despesas-2021.csv (1,723,054 linhas)
  ├─ Carregado: despesas-2022.csv (2,088,114 linhas)
  ├─ Carregado: despesas-2023.csv (2,148,364 linhas)
  ├─ Carregado: despesas-2024.csv (2,406,075 linhas)
  ├─ Carregado: despesas-2025.csv (2,387,532 linhas)
✅ CONSOLIDAÇÃO CONCLUÍDA: dados_consolidados\despesas_2021_2025.csv | Total: 10,753,139 linhas



,﻿municipio,codigo_unidade_gestora,descricao_unidade_gestora,numero_empenho,data_empenho,mes,cpf_cnpj,nome_credor,valor_empenhado,valor_liquidado,...,modalidade_licitacao,numero_obra,historico,codigo_fonte_recurso,descricao_fonte_recurso,ano_fonte,co,descricao_co,ano_referencia,arquivo_origem
0,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,1,2021-01-07,01-Janeiro,293946,BANCO DO BRASIL S/A,15.000,"14.940,75",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv
1,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,2,2021-01-07,01-Janeiro,360305003987,CAIXA ECONOMICA FEDERAL,15.000,"14.995,25",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv
2,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,3,2021-01-07,01-Janeiro,293946,BANCO DO BRASIL S/A,5.000,"4.994,5",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv
3,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,14,2021-01-07,01-Janeiro,293946,BANCO DO BRASIL S/A,1.000,"41,6",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv
4,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,15,2021-01-07,01-Janeiro,394460040950,MIN.DA FAZENDA/SEC. DO TESOURO NACIONAL,200.000,"195.259,08",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv


In [7]:
df_despesas.info()

<class 'pandas.DataFrame'>
RangeIndex: 10753139 entries, 0 to 10753138
Data columns (total 42 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   ﻿municipio                      str    
 1   codigo_unidade_gestora          float64
 2   descricao_unidade_gestora       str    
 3   numero_empenho                  int64  
 4   data_empenho                    str    
 5   mes                             str    
 6   cpf_cnpj                        int64  
 7   nome_credor                     str    
 8   valor_empenhado                 str    
 9   valor_liquidado                 str    
 10  valor_pago                      str    
 11  codigo_unidade_orcamentaria     int64  
 12  descricao_unidade_orcamentaria  str    
 13  codigo_funcao                   int64  
 14  funcao                          str    
 15  codigo_subfuncao                int64  
 16  subfuncao                       str    
 17  codigo_programa                 int6